# Analisis Eksperimen B0 — Direct LLM Baseline (Zero-Shot)

- **Konfigurasi:** `B0` (Zero-shot tanpa retrieval, tanpa few-shot, tanpa JSON schema, tanpa verifier)
- **Base Model:** `Qwen/Qwen2.5-1.5B-Instruct` | **Seed:** `42` | **Split:** `dev.jsonl` (30 query)
- **Pertanyaan Riset (`NLP_IR.pdf`):** Seberapa jauh LLM kecil tanpa konteks dokumen dan tanpa prompt terstruktur mampu menjawab pertanyaan hukum kewarisan Islam Indonesia, dan apa jenis error utamanya?

In [ ]:
import json, statistics
from pathlib import Path
from collections import Counter

ROOT = Path('..') if (Path('..') / 'runs').exists() else Path('.')
B0_DIR = ROOT / 'runs' / 'B0'

summary = json.loads((B0_DIR / 'summary.json').read_text(encoding='utf-8'))
rows = [json.loads(l) for l in (B0_DIR / 'dev_outputs.jsonl').open(encoding='utf-8') if l.strip()]

print('=== RINGKASAN B0 ===')
print(json.dumps(summary, indent=2))
print(f'Total instance: {len(rows)}')

In [ ]:
# Statistik Latensi (mean, median, p95), Token, dan Truncation
lats = sorted(r['latency_s'] for r in rows)
in_toks = [r['in_tokens'] for r in rows]
out_toks = [r['out_tokens'] for r in rows]
trunc = [r for r in rows if r['out_tokens'] >= 319]

p95_idx = min(len(lats) - 1, int(0.95 * len(lats)))
print(f'Latency  -> mean={statistics.mean(lats):.2f}s | p50={statistics.median(lats):.2f}s | p95={lats[p95_idx]:.2f}s')
print(f'In_tok   -> mean={statistics.mean(in_toks):.1f} | max={max(in_toks)}')
print(f'Out_tok  -> mean={statistics.mean(out_toks):.1f} | max={max(out_toks)}')
print(f'Truncated (>=319 output tokens): {len(trunc)}/{len(rows)} ({len(trunc)/len(rows)*100:.1f}%)')

In [ ]:
# Breakdown per Kategori Query
by_cat = {}
for r in rows:
    by_cat.setdefault(r['category'], []).append(r)

print(f"{'Kategori':18s} | {'N':>2s} | {'JSON':>4s} | {'Trunc':>5s} | {'AvgIn':>6s} | {'AvgOut':>6s} | {'AvgLat':>6s}")
print('-' * 68)
for cat, items in by_cat.items():
    jv = sum(1 for x in items if x['parsed_json'] is not None)
    tr = sum(1 for x in items if x['out_tokens'] >= 319)
    ai = sum(x['in_tokens'] for x in items) / len(items)
    ao = sum(x['out_tokens'] for x in items) / len(items)
    al = sum(x['latency_s'] for x in items) / len(items)
    print(f'{cat:18s} | {len(items):2d} | {jv:4d} | {tr:5d} | {ai:6.1f} | {ao:6.1f} | {al:5.2f}s')

In [ ]:
# Visualisasi Latensi & Output Token per Kategori (opsional jika matplotlib terpasang)
try:
    import matplotlib.pyplot as plt
    cats = list(by_cat.keys())
    avg_lats = [sum(x['latency_s'] for x in by_cat[c])/len(by_cat[c]) for c in cats]
    avg_outs = [sum(x['out_tokens'] for x in by_cat[c])/len(by_cat[c]) for c in cats]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].barh(cats, avg_lats, color='#4C78A8')
    axes[0].set_title('B0: Rata-rata Latensi per Kategori (detik)')
    axes[1].barh(cats, avg_outs, color='#F58518')
    axes[1].axvline(320, color='red', linestyle='--', label='max_new_tokens=320')
    axes[1].set_title('B0: Rata-rata Output Token per Kategori')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib belum terpasang; lewati grafik.')

In [ ]:
# Audit Kualitatif Error B0: Halusinasi Pasal & Kegagalan Abstain pada Unanswerable
def inspect_queries(qids):
    lookup = {r['query_id']: r for r in rows}
    for qid in qids:
        r = lookup.get(qid)
        if not r: continue
        print('=' * 80)
        print(f"{r['query_id']} [{r['category']} | {r['answerability']}] out_tok={r['out_tokens']} lat={r['latency_s']}s")
        print('Q:', r['query'])
        print('A:', r['raw_output'][:600])

# Contoh pasal_definisi, analisis_putusan, dan unanswerable
inspect_queries(['Q_016', 'Q_056', 'Q_082', 'Q_081', 'Q_084'])

### Kesimpulan Analisis B0 (Direct LLM Baseline)
1. **Kepatuhan Kontrak Output = 0% (`json_valid = 0.0`):** Karena B0 bersifat *zero-shot* bebas tanpa skema, output berupa prosa panjang yang tidak dapat diverifikasi secara otomatis (`claims`, `citations`, `verdict` tidak tersedia).
2. **Halusinasi Fakta Hukum Berat:** Pada `Q_016` (Pasal 182 KHI), model mengarang rujukan fiktif (*"Undang-Undang Hukum Hak Asasi Manusia dan Hak Asasi Sipil (UHHS)"*). Ini membuktikan base model 1.5B tidak menyimpan teks pasal KHI secara akurat di bobot parameternya.
3. **Tidak Ada Mekanisme Abstain (`abstain_acc = 0.0`):** Pada query di luar cakupan seperti `Q_082` (waris adat Batak Toba), model tetap menjawab panjang dan percaya diri alih-alih menolak.
4. **Latensi Tinggi & Sering Terpotong:** Rata-rata latensi **14,34 detik** (paling lambat di antara B0–B2) karena model cenderung bertele-tele hingga **16/30 (53,3%)** jawaban terpotong di batas `max_new_tokens=320`.